In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

ROOT       = Path("..").resolve()
MODEL_PATH = ROOT / "models" / "face_detector_v1_best.pt"

model = YOLO(str(MODEL_PATH))
print(f"✅ Model loaded: {MODEL_PATH}")
print(f"   Classes: {model.names}")

In [ ]:
def detect_and_show(img_path, conf=0.3, title="Detection"):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = model(img, conf=conf)[0]
    boxes   = results.boxes
    
    # Draw boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        conf_score = box.conf[0].cpu().numpy()
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (0,255,100), 2)
        cv2.putText(img_rgb, f"face {conf_score:.2f}", 
                    (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.6, (0,255,100), 2)
    
    plt.figure(figsize=(10,6))
    plt.imshow(img_rgb)
    plt.title(f"{title} — {len(boxes)} face(s) detected")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(str(ROOT / "results" / f"{title.lower().replace(' ','_')}.png"), dpi=120)
    plt.show()
    print(f"Faces detected: {len(boxes)}")

# PUT YOUR OWN PHOTO PATH HERE
detect_and_show(
    r"C:\Users\Sag\Downloads\WhatsApp Image 2026-05-13 at 6.36.42 PM (1).jpeg",
    title="Test Single Face"
)

In [ ]:
detect_and_show(r"C:\Users\Sag\Downloads\WhatsApp Image 2026-06-06 at 2.41.08 PM.jpeg", title="Test Group Face")

In [ ]:
detect_and_show(r"C:\Users\Sag\Downloads\e24bfbd855cda99e303975f2bd2a1bf43079b320-800x600.jpeg", title="Test Negative Dog")

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("Webcam started — press Q to quit")
frame_count = 0
fps_start = cv2.getTickCount()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.4, verbose=False)[0]
    boxes   = results.boxes

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        conf_score = float(box.conf[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,100), 2)
        cv2.putText(frame, f"face {conf_score:.2f}",
                    (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,100), 2)

    # FPS counter
    frame_count += 1
    elapsed = (cv2.getTickCount() - fps_start) / cv2.getTickFrequency()
    fps = frame_count / elapsed
    cv2.putText(frame, f"FPS: {fps:.1f}  Faces: {len(boxes)}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,200,255), 2)

    cv2.imshow("Face Detector V1", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Average FPS: {fps:.1f}")

In [ ]:
# Test confidence threshold effect on false positives
test_img = r"C:\Users\Sag\Downloads\e24bfbd855cda99e303975f2bd2a1bf43079b320-800x600.jpeg"  # your dog image

for conf in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    results = model(test_img, conf=conf, verbose=False)[0]
    print(f"conf={conf} → {len(results.boxes)} detection(s)")

In [ ]:
# Change this line in Cell 5
results = model(frame, conf=0.4, verbose=False, device='cpu')[0]

In [ ]:
# Check what the model actually sees on the dog image
test_img = r"C:\Users\Sag\Downloads\e24bfbd855cda99e303975f2bd2a1bf43079b320-800x600.jpeg"
results = model(test_img, conf=0.3, verbose=False)[0]

for box in results.boxes:
    print(f"Confidence : {float(box.conf[0]):.4f}")
    print(f"Box        : {box.xyxy[0].cpu().numpy().astype(int)}")

In [ ]:
# V1 Final Report
report = {
    "model"      : "YOLOv8n",
    "weights"    : "face_detector_v1_best.pt",
    "mAP50"      : 0.9188,
    "Precision"  : 0.8795,
    "Recall"     : 0.8491,
    "mAP50-95"   : 0.6273,
    "webcam_fps" : 9.0,
    "tests" : {
        "single_face"  : "PASS",
        "group_face"   : "PASS - 24 faces",
        "dog_negative" : "FAIL - conf 0.8071 (false positive)",
        "webcam"       : "PASS",
    },
    "known_issues" : [
        "Dog detected as face at 0.8071 confidence",
        "All confidence thresholds fail on dog",
        "Fix: Hard negative mining required for V2",
        "FPS 9.0 on MX250 - use CPU for better performance"
    ]
}

import json
results_path = Path("..") / "results" / "v1_final_report.json"
with open(results_path, "w") as f:
    json.dump(report, f, indent=2)

print("V1 complete. Report saved.")
print(json.dumps(report, indent=2))

In [ ]:
# V1 vs V2 comparison on dog image
model_v1 = YOLO(str(ROOT / "models" / "face_detector_v1_best.pt"))
model_v2 = YOLO(str(ROOT / "models" / "face_detector_v2_best.pt"))

dog_img = r"C:\Users\Sag\Downloads\Dog\e24bfbd855cda99e303975f2bd2a1bf43079b320-800x600.jpeg"  # same dog image as before

for conf in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    r1 = model_v1(dog_img, conf=conf, verbose=False)[0]
    r2 = model_v2(dog_img, conf=conf, verbose=False)[0]
    print(f"conf={conf} → V1: {len(r1.boxes)} detection  "
          f"V2: {len(r2.boxes)} detection  "
          f"{'✅ improved' if len(r2.boxes) < len(r1.boxes) else '='}")

# Also check exact confidence score on dog
print("\nV2 dog confidence scores:")
results_v2 = model_v2(dog_img, conf=0.1, verbose=False)[0]
for box in results_v2.boxes:
    print(f"  conf: {float(box.conf[0]):.4f}")

In [ ]:
# Full V2 test suite
model_v2 = YOLO(str(ROOT / "models" / "face_detector_v2_best.pt"))

test_cases = {
    "your_face"   : r"C:\Users\Sag\Downloads\WhatsApp Image 2026-05-13 at 6.36.42 PM (1).jpeg",
    "group_photo" : r"C:\Users\Sag\Downloads\WhatsApp Image 2026-06-06 at 2.41.08 PM.jpeg",
    "dog"         : r"C:\Users\Sag\Downloads\Dog\e24bfbd855cda99e303975f2bd2a1bf43079b320-800x600.jpeg",
    # add a cat image too if you have one
}

print("V2 Full Test Suite")
print("=" * 40)
for name, path in test_cases.items():
    results = model_v2(path, conf=0.3, verbose=False)[0]
    print(f"{name:15}: {len(results.boxes)} detection(s)")

In [ ]:
# Quick webcam test with V2 - check FPS improvement
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

model_v2 = YOLO(str(ROOT / "models" / "face_detector_v2_best.pt"))

frame_count = 0
fps_start = cv2.getTickCount()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model_v2(frame, conf=0.4, verbose=False, device='cpu')[0]
    boxes   = results.boxes

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        conf_score = float(box.conf[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,100), 2)
        cv2.putText(frame, f"face {conf_score:.2f}",
                    (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,100), 2)

    frame_count += 1
    elapsed = (cv2.getTickCount() - fps_start) / cv2.getTickFrequency()
    fps = frame_count / elapsed

    cv2.putText(frame, f"FPS: {fps:.1f}  Faces: {len(boxes)}",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,200,255), 2)
    cv2.imshow("Face Detector V2", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"V2 Average FPS: {fps:.1f}")